# Modul NLP. Klasyfikacja tematów regułami i spaCy

## Ten skrypt przypisuje kazdemu unikalnemu punktowi porzadku obrat (item) kategorie tematyczną. 
Uzte zostaly tylko spaCy i pandas. Działa w pełni lokalnie


### Sposob działania:
1. spaCy zmienai tytul na liste lematów (podstawowe formy słów) np. 'zabytkow' -> 'zabytek'
2. dla kazdej kategorii mamy slownik slow kluczowych. Slowo kluczowe pasuje do lematum jesli lemat od niego sie zaczyna. Np. 'energ' -> 'energia', 'energetyczny'
3. Kazde trafienie to jeden punkt dla kategorii. Wygrywa kategoria z najwieksza liczba puntkow, druga w kolejnsoci staje sie etykieta dodatkową
4. Brak zadnej kategorii -> etykita 'inne'


Czesz A: wgrywanie danych

In [92]:
import hashlib
import json
import re
from pathlib import Path

import pandas as pd
import spacy

ROOT = Path.cwd().parent if Path.cwd().name in ("notebook", "notebooks") else Path.cwd()
INTERIM = ROOT / 'code' / "data" / "interim"
LABELING = ROOT / "data" / "labeling"
RESULTS = ROOT / "results"
LABELING.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(exist_ok=True)

votings = pd.read_csv("code/data/interim/votings_text.csv", dtype={"print_numbers": str})
#do poprwaki votings, aby działało dla wszystkich a nie tylko lokalnie
print("Liczba głosowań:", len(votings))
votings[["title", "topic", "description", "print_title"]].head(3)

Liczba głosowań: 5306


,title,topic,description,print_title
0,Wybór Marszałka Sejmu RP,Wybór Marszałka Sejmu RP,wybór Marszałka Sejmu,NaN
1,Pkt. 2 Poselskie projekty uchwał w sprawie ust...,NaN,w sprawie ustalenia liczby wicemarszałków Sejmu,Poselski projekt uchwały w sprawie ustalenia l...
2,Pkt. 3 Wybór Wicemarszałków Sejmu RP (druki nr...,Głosowanie nad kandydaturą Posła Krzysztofa Bo...,w sprawie wyboru Krzysztofa Bosaka na wicemars...,NaN


### Krok A2. Normalizacja tytyłu punku porzadku obrad
Większość głosowań to poprawki i wnioski do jednej ustawy, więc tematem jest ustawa, a nie pojedyncza poprawka. Z tytułu usuwamy numer punktu ("Pkt. 33"), nawias z numerami druków i końcówkę z etapem czytania. Wynik to kolumna `item`. Etap czytania zapisujemy osobno w kolumnie `stage`. Dzięki temu wszystkie poprawki do jednej ustawy w tym samym czytaniu mają identyczny `item` i klasyfikujemy je jeden raz.

Wyjątek: głosowania "Głosowanie proceduralne dotyczące druku N" nie mówią nic o treści, ale mają tytuł druku (`print_title`), który opisuje projekt. W ich przypadku `item` bierzemy z tytułu druku.

In [62]:
STAGE = re.compile(r"[-–]\s*(pierwsze|drugie|trzecie)\s+czytanie", re.IGNORECASE)
HEADER = re.compile(r"^\s*\d+\.\s*posiedzenie\s+Sejmu", re.IGNORECASE)
ORG = re.compile(
    r"(?:zmian\w*\s+w\s+skład\w*\s+osobow|do\s+składu\s+osobowego|wybór\s+uzupełniaj|wybór\s+(?:wice)?marszałk|wybór\s+skład)",
    re.IGNORECASE,
)


def extract_stage(title):
    if not isinstance(title, str):
        return None
    m = STAGE.search(title)
    if m:
        return m.group(1).lower()
    return None


def normalize_item(title):
    if not isinstance(title, str):
        return None
    t = re.sub(r"^\s*Pkt\.\s*\d+\s*", "", title)
    t = STAGE.sub("", t)
    t = re.sub(r"\s*\(druk[^)]*\)", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+", " ", t).strip(" -–;:,")
    if t == "":
        return None
    return t


title = votings["title"].fillna("")
votings["stage"] = votings["title"].apply(extract_stage)
votings["is_header"] = title.str.match(HEADER.pattern, case=False)
votings["item"] = votings["title"].apply(normalize_item)

# głosowania proceduralne dotyczące druku: temat bierzemy z tytułu druku
proc_print = title.str.contains(r"^Głosowanie proceduralne", case=False, regex=True) & votings["print_title"].notna()
votings.loc[proc_print, "item"] = votings.loc[proc_print, "print_title"].apply(normalize_item)
print("Głosowania proceduralne z tytułem druku:", int(proc_print.sum()))

votings[["title", "item", "stage"]].sample(8, random_state=1)

Głosowania proceduralne z tytułem druku: 200


,title,item,stage
4699,65. posiedzenie Sejmu Rzeczypospolitej Polskie...,65. posiedzenie Sejmu Rzeczypospolitej Polskie...,None
3774,Pkt. 5 Sprawozdanie Komisji o rządowym projekc...,Sprawozdanie Komisji o rządowym projekcie usta...,None
3135,Pkt. 1 Sprawozdanie Komisji o rządowym projekc...,Sprawozdanie Komisji o rządowym projekcie usta...,trzecie
3756,Pkt. 12 Sprawozdanie Komisji o rządowym projek...,Sprawozdanie Komisji o rządowym projekcie usta...,None
4144,Pkt. 30 Sprawozdanie Ministra Spraw Wewnętrzny...,Sprawozdanie Ministra Spraw Wewnętrznych i Adm...,None
4344,Pkt. 16 Sprawozdanie Komisji o uchwale Senatu ...,Sprawozdanie Komisji o uchwale Senatu w sprawi...,None
5257,Pkt. 5 Sprawozdanie Komisji o rządowym projekc...,Sprawozdanie Komisji o rządowym projekcie usta...,None
2107,Pkt. 22 Sprawozdanie Komisji o rządowym oraz p...,Sprawozdanie Komisji o rządowym oraz poselskic...,None


### Krok A3: głosowania bez treści merytorycznej

Za proceduralne i organizacyjne uznajemy nagłówki posiedzeń, zmiany w składach komisji, wybory uzupełniające i wybory wicemarszałków oraz wiersze bez tytułu. Dostają kategorię `proceduralne_organizacyjne` z reguły i nie trafią do klasyfikatora. Reguły są celowo wąskie: lepiej zostawić wątpliwy przypadek klasyfikatorowi niż zgubić głosowanie merytoryczne.

In [63]:
org = title.str.contains(ORG.pattern, case=False, regex=True)
proc_no_content = title.str.contains(r"^\s*Głosowanie proceduralne", case=False, regex=True) & votings["print_title"].isna()
votings["rule_category"] = None
mask = votings["is_header"] | org | votings["item"].isna() | proc_no_content
votings.loc[mask, "rule_category"] = "proceduralne_organizacyjne"
votings["needs_nlp"] = votings["rule_category"].isna()

print("Proceduralne z reguły:", int(mask.sum()), "| do klasyfikacji:", int(votings["needs_nlp"].sum()))
print("Przykłady wykluczonych tytułów:")
print(votings.loc[mask, "title"].drop_duplicates().head(8).to_string())

Proceduralne z reguły: 573 | do klasyfikacji: 4733
Przykłady wykluczonych tytułów:
0                              Wybór Marszałka Sejmu RP
2     Pkt. 3 Wybór Wicemarszałków Sejmu RP (druki nr...
9     Pkt. 5 Wybór składu osobowego Komisji Regulami...
11    1. posiedzenie Sejmu w dniu 13 i 14 listopada ...
13    1. posiedzenie Sejmu w dniach 13, 14, 21, 22 i...
17    Pkt. 8 Wybór składów osobowych komisji sejmowy...
18    Pkt. 9 Wybór składu osobowego Komisji do Spraw...
19    Pkt. 10 Wybór składu osobowego Komisji Etyki P...


### Krok A4: typ głosowania regułami

Typ głosowania (poprawka, wniosek mniejszości, wniosek formalny, ...) wyciągamy z pól `topic` i `description`. To czysto formalna cecha tekstu, więc nie potrzebuje klasyfikatora. Jest ważna analitycznie, bo dyscyplina w głosowaniu nad poprawką i nad całością ustawy może się różnić. Pierwsza reguła, która pasuje, wygrywa, więc kolejność ma znaczenie. Reguły dla głosowań nad całością ustawy to na razie przypuszczenie, bo nie widzieliśmy wszystkich sformułowań. Komórka diagnostyczna pokazuje najczęstsze wzorce tematów w kategorii `inne`. Dopisz dla nich reguły i uruchom ponownie.

In [64]:
VOTE_TYPES = [
    ("poprawka", r"^\s*poprawk"),
    ("wniosek_mniejszosci", r"wniosek\s+mniejszości"),
    ("skrocenie_terminu", r"skróceni\w*\s+terminu"),
    ("odroczenie_przerwa", r"(odroczeni|przerw)"),
    ("przyjecie_wniosku", r"przyjęci\w*\s+wniosku"),
    ("odrzucenie", r"odrzuceni"),
    ("uchwala_senatu", r"uchwał\w*\s+senatu"),
    ("calosc", r"(?:całoś\w*|uchwaleni\w*|przyjęci\w*\s+(?:ustawy|projektu|uchwały))"),
]


def vote_type(row):
    text = ""
    for field in (row["topic"], row["description"]):
        if isinstance(field, str):
            text = text + " " + field
    for name, pattern in VOTE_TYPES:
        if re.search(pattern, text, re.IGNORECASE):
            return name
    return "inne"


votings["vote_type"] = votings.apply(vote_type, axis=1)
print(votings["vote_type"].value_counts())

# diagnostyka: najczęstsze wzorce tematów, które nie pasują do żadnej reguły (cyfry zastąpione przez N)
unmatched = votings[votings["vote_type"] == "inne"]
patterns = (
    unmatched["topic"].fillna("(brak tematu)")
    .str.lower().str.replace(r"\d+", "N", regex=True)
    .value_counts()
)
patterns.head(25)

vote_type
poprawka               2534
calosc                  798
inne                    486
wniosek_mniejszosci     425
skrocenie_terminu       333
odrzucenie              261
przyjecie_wniosku       244
odroczenie_przerwa      224
uchwala_senatu            1
Name: count, dtype: int64


topic
głosowanie kworum                                                                                                                                                               57
wniosek o przystąpienie do trzeciego czytania bez kierowania projektu do komisji                                                                                                41
wnioski mniejszości nr N i N                                                                                                                                                    38
wniosek o uzupełnienie porządku dziennego.                                                                                                                                      32
wnioski mniejszości nr N-N                                                                                                                                                      29
ustalenie czasów debat                                                                             

### Krok A5: unikalne punkty porządku obrad

Tylko głosowania z `needs_nlp = True` mają trafić do klasyfikatora. Grupujemy je po `item`. Kolumna `item_id` to krótki skrót tekstu, który służy jako stabilny klucz przy łączeniu etykiet z głosowaniami. Uwaga: skrót zależy od tekstu `item`, więc po utworzeniu arkusza do oznaczania (krok A6) nie zmieniaj reguł normalizacji z kroku A2. Zmiana tekstu zmienia skrót, a etykiety przestają pasować do punktów.

Policz, ile jest unikalnych punktów w porównaniu z liczbą głosowań. To bezpośrednio określa pracę przy oznaczaniu i koszt klasyfikacji.

In [65]:
items = (
    votings[votings["needs_nlp"]]
    .groupby("item")
    .agg(n_votings=("glosowanie_id", "size"), n_sittings=("sitting", "nunique"), first_date=("date", "min"))
    .reset_index()
    .sort_values("n_votings", ascending=False)
    .reset_index(drop=True)
)

item_ids = []
for text in items["item"]:
    item_ids.append(hashlib.md5(text.encode("utf-8")).hexdigest()[:10])
items["item_id"] = item_ids
assert items["item_id"].is_unique, "kolizja skrótów, wydłuż item_id"

votings = votings.merge(items[["item", "item_id"]], on="item", how="left")

n_v = int(votings["needs_nlp"].sum())
print("Unikalne punkty do klasyfikacji:", len(items), "| głosowania, które pokrywają:", n_v)
print("Mediana głosowań na punkt:", items["n_votings"].median(), "| maksimum:", items["n_votings"].max())
print("Udział głosowań w 100 największych punktach:", round(items["n_votings"].head(100).sum() / n_v, 3))
items.head(10)

Unikalne punkty do klasyfikacji: 1333 | głosowania, które pokrywają: 4733
Mediana głosowań na punkt: 1.0 | maksimum: 149
Udział głosowań w 100 największych punktach: 0.455


,item,n_votings,n_sittings,first_date,item_id
0,Sprawozdanie Komisji o rządowym projekcie usta...,149,1,2024-12-06T09:32:32,f89c2d855d
1,Sprawozdanie Komisji o rządowym projekcie usta...,116,3,2024-10-01T19:40:44,3bb21732e7
2,Sprawozdanie Komisji o rządowym projekcie usta...,99,1,2025-12-05T15:18:07,e6aee776c2
3,Sprawozdanie Komisji o rządowym projekcie usta...,54,1,2026-10-22T18:00:40,7d7265e51a
4,Sprawozdanie Komisji o rządowym projekcie usta...,51,3,2025-11-21T10:15:41,521038377e
5,Sprawozdanie Komisji o rządowym projekcie usta...,50,1,2026-03-11T21:38:36,891eda118f
6,Sprawozdanie Komisji o przedstawionym przez Pr...,43,1,2026-09-18T10:27:11,15778c6ad5
7,Sprawozdanie Komisji o rządowym projekcie usta...,41,1,2026-05-29T10:02:41,cb554a6fdf
8,Sprawozdanie Komisji o rządowym projekcie usta...,39,1,2026-10-08T19:28:15,7a52053fc6
9,Sprawozdanie Komisji o rządowym projekcie usta...,35,1,2026-10-08T19:53:45,84abde2ae7


### Krok A6: kategorie i arkusz do ręcznego oznaczania

Poniżej robocza taksonomia: klucz kategorii i krótka definicja. Zmieniaj ją swobodnie, zanim zaczniesz oznaczać, ale potem nie zmieniaj znaczenia kategorii bez zapisania tego w dokumentacji. Klucze są zapisane bez polskich znaków, bo służą jako identyfikatory w danych. Kategoria `proceduralne_organizacyjne` jest przypisywana wyłącznie regułami, a `inne` oznacza treść merytoryczną, która nie pasuje do żadnej kategorii.

Dalej losujemy 100 unikalnych punktów (ziarno losowania jest stałe, więc wynik jest powtarzalny). Pierwsze 40 to zbiór `tune`, na którym poprawiasz słownik, a pozostałe 60 to zbiór `test`, którego nie wolno oglądać przy strojeniu. Losowanie jest jednostajne po punktach, więc ocena dotyczy jakości klasyfikacji typowego punktu, a nie typowego głosowania.

Zasady oznaczania: w kolumnie `label_main` wpisz klucz kategorii, a w `label_secondary` opcjonalnie drugi, jeśli temat wyraźnie należy do dwóch obszarów. W `notes` zapisuj wątpliwości. Najlepiej zrób drugie przejście po kilku dniach w kolumnie `label_main_pass2`, nie patrząc na pierwsze etykiety: zgodność między przejściami to górna granica tego, czego można oczekiwać od klasyfikatora. Arkusz nie jest nadpisywany, jeśli już istnieje.

Po oznaczeniu zapisz plik jako `results/gold_labels.csv`. Katalog `results/` jest śledzony przez git, a ręczne etykiety są cennym zasobem, którego nie da się odtworzyć skryptem.

In [66]:
CATEGORIES = {
    "energia_klimat": "Energetyka, ceny energii i paliw, polityka klimatyczna i emisje.",
    "zdrowie": "System ochrony zdrowia, leki, zdrowie publiczne i profilaktyka (w tym alkohol i tytoń).",
    "polityka_spoleczna": "Emerytury i renty, świadczenia, pomoc społeczna, rodzina i dzieci, niepełnosprawność.",
    "budzet_finanse": "Budżet państwa, podatki, system finansowy i bankowy, finanse publiczne, ubezpieczenia.",
    "gospodarka_praca": "Regulacje gospodarcze i handlowe, przedsiębiorczość, rynek pracy i prawo pracy, konsumenci.",
    "rolnictwo_srodowisko": "Rolnictwo, rybołówstwo, leśnictwo, wieś, ochrona środowiska i przyrody, woda i odpady.",
    "edukacja_nauka_kultura": "Szkolnictwo, nauka i szkolnictwo wyższe, kultura, dziedzictwo i zabytki, sport.",
    "sprawiedliwosc_prawa": "Sądy, prokuratura, prawo karne i cywilne, prawa i wolności obywatelskie, równość.",
    "bezpieczenstwo_obrona": "Wojsko, obrona, policja i służby specjalne, straż graniczna, migracje i granice.",
    "zagraniczne_ue": "Umowy międzynarodowe, ratyfikacje, Unia Europejska, stosunki z innymi państwami.",
    "ustroj_administracja": "Ustrój państwa, wybory, samorząd terytorialny, administracja, instytucje państwowe.",
    "infrastruktura_transport": "Transport, drogi, kolej, lotnictwo, budownictwo, planowanie przestrzenne, mieszkalnictwo.",
    "technologie_media": "Cyfryzacja, telekomunikacja, cyberbezpieczeństwo, sztuczna inteligencja, media.",
    "inne": "Treść merytoryczna, która nie pasuje do żadnej z powyższych kategorii albo jest zbyt niejasna.",
    "proceduralne_organizacyjne": "Nadawane regułami: nagłówki posiedzeń, wnioski formalne bez przedmiotu, zmiany w składach komisji.",
}
CATEGORY_NAMES = list(CATEGORIES.keys())

for key, description in CATEGORIES.items():
    print(f"{key:28s} {description}")

energia_klimat               Energetyka, ceny energii i paliw, polityka klimatyczna i emisje.
zdrowie                      System ochrony zdrowia, leki, zdrowie publiczne i profilaktyka (w tym alkohol i tytoń).
polityka_spoleczna           Emerytury i renty, świadczenia, pomoc społeczna, rodzina i dzieci, niepełnosprawność.
budzet_finanse               Budżet państwa, podatki, system finansowy i bankowy, finanse publiczne, ubezpieczenia.
gospodarka_praca             Regulacje gospodarcze i handlowe, przedsiębiorczość, rynek pracy i prawo pracy, konsumenci.
rolnictwo_srodowisko         Rolnictwo, rybołówstwo, leśnictwo, wieś, ochrona środowiska i przyrody, woda i odpady.
edukacja_nauka_kultura       Szkolnictwo, nauka i szkolnictwo wyższe, kultura, dziedzictwo i zabytki, sport.
sprawiedliwosc_prawa         Sądy, prokuratura, prawo karne i cywilne, prawa i wolności obywatelskie, równość.
bezpieczenstwo_obrona        Wojsko, obrona, policja i służby specjalne, straż graniczna, migracje i 

In [67]:
SHEET = LABELING / "items_to_label.csv"

if SHEET.exists():
    print("Arkusz już istnieje, nie nadpisuję:", SHEET)
else:
    n = min(100, len(items))
    n_tune = min(40, n // 2)
    sample = items.sample(n=n, random_state=42).reset_index(drop=True)
    sample["split"] = ["tune"] * n_tune + ["test"] * (n - n_tune)
    for column in ("label_main", "label_secondary", "label_main_pass2", "notes"):
        sample[column] = ""
    sheet_columns = ["item_id", "split", "item", "n_votings", "label_main", "label_secondary", "label_main_pass2", "notes"]
    sample[sheet_columns].to_csv(SHEET, index=False)
    print("Zapisano arkusz:", SHEET, "| wierszy:", n, "| tune:", n_tune, "| test:", n - n_tune)

Arkusz już istnieje, nie nadpisuję: /Users/przemekturski/Library/CloudStorage/OneDrive-Osobisty/Pliki studia/5. Ekonometria/sejm-network-analysis/data/labeling/items_to_label.csv


## Część B: klasyfikacja regułami i spaCy

Metoda w czterech zdaniach:
1. spaCy zamienia tytuł na listę lematów, np. "zabytków" na "zabytek".
2. Dla każdej kategorii mamy słownik słów kluczowych. Słowo kluczowe pasuje do lematu, jeśli lemat od niego zaczyna się (np. `energ` pasuje do `energia`, `energetyczny`).
3. Każde trafienie to jeden punkt dla kategorii. Wygrywa kategoria z największą liczbą punktów, a druga w kolejności staje się etykietą dodatkową.
4. Gdy żadna kategoria nie dostała punktu, etykieta to `inne`.

Cała praca polega na poprawianiu słownika. Poprawiasz go, patrząc wyłącznie na zbiór `tune`. Zbiór `test` oglądasz raz, na samym końcu.

### Krok B1: ładowanie modelu

Parser i rozpoznawanie nazw własnych wyłączamy, bo do lematyzacji nie są potrzebne, a wyłączenie przyspiesza pracę. Sprawdziłem na przykładowych tytułach, że lematy są wtedy identyczne.

In [68]:
nlp = spacy.load("pl_core_news_md", disable=["parser", "ner"])
print("Załadowano model. Komponenty:", nlp.pipe_names)

Załadowano model. Komponenty: ['tok2vec', 'morphologizer', 'lemmatizer', 'tagger', 'attribute_ruler']


### Krok B2: jak spaCy widzi tytuł

Obejrzyj na jednym przykładzie, jak tytuł zamienia się w tokeny i lematy. Wykorzystamy tylko kolumnę `lemma`. Lematyzator czasem się myli (np. zostawia formę "uchwale"), dlatego słowa kluczowe będą krótkimi początkami słów, a nie pełnymi formami.


In [69]:
example = "Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o ochronie zabytków i opiece nad zabytkami"
doc = nlp(example)

for token in doc:
    print(f"{token.text:15s} {token.lemma_:15s} {token.pos_}")

Sprawozdanie    sprawozdanie    NOUN
Komisji         komisja         NOUN
o               o               ADP
rządowym        rządowy         ADJ
projekcie       projekt         NOUN
ustawy          ustawa          NOUN
o               o               ADP
zmianie         zmiana          NOUN
ustawy          ustawa          NOUN
o               o               ADP
ochronie        ochrona         NOUN
zabytków        zabytek         NOUN
i               i               CCONJ
opiece          opieka          NOUN
nad             nad             ADP
zabytkami       zabytek         NOUN


### Krok B3: lematy dla wszystkich punktów

Funkcja `get_lemmas` zwraca listę lematów małymi literami, bez znaków interpunkcji i spacji. Liczymy lematy raz dla wszystkich punktów i trzymamy je w kolumnie `lemmas`, żeby kolejne uruchomienia słownika były szybkie.

In [70]:
BOILERPLATE_PHRASES = ["z prac nad"]


def get_lemmas(text):
    for phrase in BOILERPLATE_PHRASES:
        text = text.replace(phrase, " ")
    doc = nlp(text)
    lemmas = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemmas.append(token.lemma_.lower())
    return lemmas


lemmas_list = []
for text in items["item"]:
    lemmas_list.append(get_lemmas(text))
items["lemmas"] = lemmas_list

items[["item", "lemmas"]].head(3)

,item,lemmas
0,Sprawozdanie Komisji o rządowym projekcie usta...,"[sprawozdanie, komisja, o, rządowy, projekt, u..."
1,Sprawozdanie Komisji o rządowym projekcie usta...,"[sprawozdanie, komisja, o, rządowy, projekt, u..."
2,Sprawozdanie Komisji o rządowym projekcie usta...,"[sprawozdanie, komisja, o, rządowy, projekt, u..."


### Krok B4: słownik kategorii

Zasady pisania słownika:
1. Słowo kluczowe to początek lematu małymi literami, np. `podat` pasuje do `podatek`, `podatnik`, `podatkowy`. Uwaga: lemat to forma podstawowa (mianownik liczby pojedynczej), w której samogłoska często znika w odmianie. Dla `zabytków` lematem jest `zabytek`, więc słowo kluczowe to `zabyt`, a nie `zabytk`. Podobnie `uczeń`, `więzień`, `węgiel`, `zasiłek`, `budynek`. Komórka w kroku B6 wypisuje słowa kluczowe, które nie pasują do żadnego punktu, i pomaga wyłapać takie pomyłki.
2. Słowo kluczowe zakończone znakiem `$` musi być całym lematem, np. `gaz$` pasuje do `gaz`, ale nie do `gazeta`. Używaj tego dla krótkich słów, które łatwo pomylić.
3. Unikaj słów, które występują w tytułach wszystkich ustaw (`projekt`, `ustawa`, `zmiana`, `komisja`).
4. Remis rozstrzyga kolejność kategorii w tym słowniku: wyżej wpisana wygrywa.
5. Kategorie `inne` i `proceduralne_organizacyjne` nie mają słów kluczowych. Pierwsza jest brakiem trafień, druga powstaje regułami w kroku A3.

To pierwsza wersja, napisana bez oglądania twoich etykiet. Rozbudowuj ją w pętli: uruchom ocenę na `tune` (krok B7), obejrzyj błędy (krok B7) i punkty bez trafień (krok B6), dopisz słowa, uruchom ponownie od tej komórki. Po każdej większej zmianie podnieś `RULES_VERSION`.

In [83]:
RULES_VERSION = "v2"

RULES = {
    "energia_klimat": [
        "energ", "paliw", "gaz$", "gazow", "ciepł", "elektr", "prąd$", "węgi", "węgl", "naft", "ropa$",
        "klimat", "emisj", "odnawialn", "wiatrow", "fotowoltaik", "atom", "jądrow", "górnic", "górnict",
    ],
    "zdrowie": [
        "zdrow", "leczni", "lekarsk", "lekarz", "lek$", "szpital", "pacjent", "farmac", "aptek",
        "medyczn", "medycyn", "choro", "ratownict", "sanitarn", "epidemi", "szczepi", "alkohol",
        "trzeźw", "tyto", "papieros", "narkoman", "pielęgniar", "uzależni", "nfz$",
    ],
    "polityka_spoleczna": [
        "emerytal", "emerytur", "emeryt", "renta$", "rentow", "zasił", "społeczn", "rodzin",
        "dziec", "niepełnospraw", "opiekuńcz", "opiekun", "senior", "alimentac", "macierzyńsk",
        "rodzicielsk", "żłob",
    ],
    "budzet_finanse": [
        "budżet", "podat", "finans", "bank", "skarb", "celn", "akcyz", "vat$", "ubezpiecz", "kredyt",
        "pożyczk", "obligacj", "walut", "zadłużen", "dług$", "deficyt", "rachunkow", "audyt",
    ],
    "gospodarka_praca": [
        "gospodar", "przedsiębior", "handel", "handl", "konsument", "prac", "pracownik", "pracodawc",
        "zatrudni", "wynagrodz", "płac", "rynek$", "rynk", "konkurenc", "spółk", "zamówieni",
        "przemysł", "turystyk", "rzemiosł", "hazard", "upadłoś", "restrukturyz", "produkt"
    ],
    "rolnictwo_srodowisko": [
        "roln", "wieś$", "wiejsk", "leśn", "las$", "łowiect", "rybo", "rybac", "środowisk", "przyrod",
        "zwierz", "wodn", "woda$", "wodoci", "odpad", "recykling", "hodowl", "żywnoś", "pestycyd", "zboż",
    ],
    "edukacja_nauka_kultura": [
        "edukac", "oświat", "szkol", "nauczyciel", "uczeń$", "uczni", "student", "uczeln", "wyższ", "nauk",
        "kultur", "zabyt", "muze", "bibliotek", "bibliotk", "archiw", "teatr", "film", "sport", "dziedzictw",
        "artyst", "autorsk", "przedszkol", "kształcen", "akademick", "uczc", "upamiętni", "rocznic", "ustanow", "uniwersytet",
    ],
    "sprawiedliwosc_prawa": [
        "sąd$", "sądow", "sądownict", "sędzi", "prokurat", "karn", "kara$", "przestęp", "cywiln",
        "adwokat", "notariusz", "komornik", "więzien", "więzień$", "więźni", "wykroczeni", "trybunał",
        "obywatelsk", "wolnoś", "równoś", "dyskrymin", "rodo$", "antykorupcyjn", "ekstradycj", "mniejszoś" , "etniczn"
    ],
    "bezpieczenstwo_obrona": [
        "obron", "wojsk", "wojn", "wojen", "żołnierz", "zbrojn", "policj", "wywiad", "kontrwywiad",
        "bezpieczeńst", "terroryst", "antyterror", "graniczn", "granic", "migrac", "cudzoziem",
        "azyl", "uchodźc", "straż$", "pożarn", "broń$", "amunicj", "mundurow", "abw$",
    ],
    "zagraniczne_ue": [
        "międzynarodow", "ratyfik", "konwencj", "traktat", "unia$", "unijn", "europejsk",
        "zagranicz", "dyplomat", "konsularn", "ambasad", "nato$", "protokół$", "rezolucj"
    ],
    "ustroj_administracja": [
        "samorząd", "gmin", "powiat", "województw", "wyborcz", "wyborc", "referend", "administrac",
        "urząd", "urzęd", "konstytuc", "ustrój", "regulamin", "sejm$", "senat$", "parlament", "prezydent$",
        "rząd$", "minist", "partia$", "partyjn", "kadencj", "mandat", "poseł$", "posłow", "senator",
        "terytorialn", "stołeczn", "wstępn", "powoła"
    ],
    "infrastruktura_transport": [
        "transport", "drog", "kolej$", "kolejow", "lotnic", "lotnisk", "lotnicz", "morsk", "port$", "autostrad",
        "samochod", "pojazd", "kierowc", "ruch$", "budowl", "budownict", "budyn", "mieszkani",
        "mieszkaln", "nieruchomo", "przestrzenn", "zagospodarowani", "infrastruktur", "inwestycj",
        "geodez", "kataster", "metro$",
    ],
    "technologie_media": [
        "cyfrow", "cyfryz", "teleko", "elektroniczn", "internet", "informatyz", "informatycz", "cyber",
        "sztuczn", "algorytm", "media$", "medium$", "medialn", "telewizj", "radiofon", "prasow", "prasa$", "radio$",
        "dziennikarz", "platform", "oprogramowan",
    ],
}

print("Kategorie ze słownikiem:", len(RULES), "| liczba słów kluczowych:", sum(len(v) for v in RULES.values()))

Kategorie ze słownikiem: 13 | liczba słów kluczowych: 294


### Krok B5: punktacja i wybór etykiety

`lemma_matches` sprawdza jedno słowo kluczowe z jednym lematem. `score_lemmas` liczy punkty: każdy lemat daje kategorii najwyżej jeden punkt (po pierwszym pasującym słowie kluczowym przerywamy). Dzięki temu długa lista słów kluczowych nie premiuje kategorii za samą liczbę wpisów.

`choose_label` wybiera kategorię z największą liczbą punktów. Sortowanie w Pythonie jest stabilne, więc przy remisie wygrywa kategoria wpisana wyżej w słowniku. `confidence` to udział punktów zwycięzcy w sumie wszystkich punktów, czyli liczba od 0 do 1. To zwykły wskaźnik jednoznaczności, a nie prawdopodobieństwo.

In [73]:
def lemma_matches(lemma, keyword):
    if keyword.endswith("$"):
        return lemma == keyword[:-1]
    return lemma.startswith(keyword)


def score_lemmas(lemmas):
    scores = {}
    for category, keywords in RULES.items():
        points = 0
        for lemma in lemmas:
            for keyword in keywords:
                if lemma_matches(lemma, keyword):
                    points = points + 1
                    break  # jeden lemat daje kategorii najwyżej jeden punkt
        scores[category] = points
    return scores


def choose_label(scores):
    ordered = sorted(scores.items(), key=lambda pair: -pair[1])
    best_category, best_points = ordered[0]
    second_category, second_points = ordered[1]

    if best_points == 0:
        return "inne", None, 0.0

    secondary = None
    if second_points > 0:
        secondary = second_category

    total = sum(scores.values())
    confidence = best_points / total
    return best_category, secondary, confidence


# test na jednym przykładzie
test_lemmas = get_lemmas("o zmianie ustawy o ochronie zabytków i opiece nad zabytkami")
test_scores = score_lemmas(test_lemmas)
print({k: v for k, v in test_scores.items() if v > 0})
print(choose_label(test_scores))

{'edukacja_nauka_kultura': 2}
('edukacja_nauka_kultura', None, 1.0)


### Krok B6: narzędzia diagnostyczne

`explain` pokazuje, które lematy zadecydowały o wyniku. Użyj go, gdy klasyfikator pomyli się i nie wiesz dlaczego. Potem uruchamiamy klasyfikację na wszystkich punktach i wypisujemy najczęstsze punkty z etykietą `inne`, czyli te, na których dopisanie słów da największy efekt (sortujemy je po liczbie głosowań).

In [74]:
def explain(text):
    lemmas = get_lemmas(text)
    print("Lematy:", lemmas)
    for category, keywords in RULES.items():
        for lemma in lemmas:
            for keyword in keywords:
                if lemma_matches(lemma, keyword):
                    print(f"  {category}: lemat '{lemma}' pasuje do słowa kluczowego '{keyword}'")
                    break


explain("o zmianie ustawy o wychowaniu w trzeźwości i przeciwdziałaniu alkoholizmowi oraz ustawy o radiofonii i telewizji")

Lematy: ['o', 'zmiana', 'ustawa', 'o', 'wychowanie', 'w', 'trzeźwość', 'i', 'przeciwdziałaniu', 'alkoholizm', 'oraz', 'ustawa', 'o', 'radiofonia', 'i', 'telewizja']
  zdrowie: lemat 'trzeźwość' pasuje do słowa kluczowego 'trzeźw'
  zdrowie: lemat 'alkoholizm' pasuje do słowa kluczowego 'alkohol'
  technologie_media: lemat 'radiofonia' pasuje do słowa kluczowego 'radiofon'
  technologie_media: lemat 'telewizja' pasuje do słowa kluczowego 'telewizj'


In [75]:
labels = []
secondary_labels = []
confidences = []
for lemmas in items["lemmas"]:
    scores = score_lemmas(lemmas)
    label, secondary, confidence = choose_label(scores)
    labels.append(label)
    secondary_labels.append(secondary)
    confidences.append(confidence)

items["label"] = labels
items["secondary_label"] = secondary_labels
items["confidence"] = confidences

print("Rozkład etykiet (liczba punktów):")
print(items["label"].value_counts())
print()
print("Punkty bez trafień:", int((items["label"] == "inne").sum()), "z", len(items))

Rozkład etykiet (liczba punktów):
label
budzet_finanse              190
ustroj_administracja        180
edukacja_nauka_kultura      168
sprawiedliwosc_prawa        148
inne                        108
gospodarka_praca            106
energia_klimat               74
infrastruktura_transport     67
zdrowie                      66
bezpieczenstwo_obrona        63
zagraniczne_ue               52
rolnictwo_srodowisko         50
polityka_spoleczna           49
technologie_media            12
Name: count, dtype: int64

Punkty bez trafień: 108 z 1333


In [76]:
# najważniejsze punkty bez trafień: dopisz dla nich słowa kluczowe
no_hits = items[items["label"] == "inne"].sort_values("n_votings", ascending=False)
for _, row in no_hits.head(25).iterrows():
    print(row["n_votings"], "|", row["item"][:130])
    print("     lematy:", row["lemmas"][:14])

116 | Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o szczególnych rozwiązaniach związanych z usuwaniem skutków powo
     lematy: ['sprawozdanie', 'komisja', 'o', 'rządowy', 'projekt', 'ustawa', 'o', 'zmiana', 'ustawa', 'o', 'szczególny', 'rozwiązanie', 'związać', 'z']
41 | Sprawozdanie Komisji o rządowym projekcie ustawy - Przepisy wprowadzające ustawę o statusie osoby najbliższej w związku i umowie o
     lematy: ['sprawozdanie', 'komisja', 'o', 'rządowy', 'projekt', 'ustawa', 'przepis', 'wprowadzać', 'ustawa', 'o', 'status', 'osoba', 'bliski', 'w']
35 | Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o usługach hotelarskich oraz usługach pilotów wycieczek i przewo
     lematy: ['sprawozdanie', 'komisja', 'o', 'rządowy', 'projekt', 'ustawa', 'o', 'zmiana', 'ustawa', 'o', 'usługa', 'hotelarski', 'oraz', 'usługa']
19 | Sprawozdanie Komisji o rządowym projekcie ustawy o szczególnych środkach ochrony osób uczestniczących w debacie publicznej
     lem

In [77]:
for category, keywords in RULES.items():
    unused = []
    for keyword in keywords:
        found = False
        for lemmas in items["lemmas"]:
            for lemma in lemmas:
                if lemma_matches(lemma, keyword):
                    found = True
                    break
            if found:
                break
        if not found:
            unused.append(keyword)
    if len(unused) > 0:
        print(category + ":", unused)

energia_klimat: ['prąd$', 'fotowoltaik', 'atom']
zdrowie: ['szpital', 'aptek', 'medycyn', 'sanitarn', 'epidemi', 'szczepi', 'papieros', 'uzależni', 'nfz$']
polityka_spoleczna: ['rentow', 'zasił', 'opiekun', 'senior', 'alimentac', 'macierzyńsk', 'rodzicielsk', 'żłob']
budzet_finanse: ['vat$', 'zadłużen', 'deficyt']
gospodarka_praca: ['pracodawc', 'rzemiosł']
rolnictwo_srodowisko: ['las$', 'łowiect', 'rybac', 'wodoci', 'hodowl', 'pestycyd', 'zboż']
edukacja_nauka_kultura: ['uczeń$', 'uczni', 'uczeln', 'muze', 'bibliotek', 'bibliotk', 'teatr', 'film', 'przedszkol']
sprawiedliwosc_prawa: ['notariusz', 'więzień$', 'więźni', 'dyskrymin', 'rodo$', 'ekstradycj']
bezpieczenstwo_obrona: ['terroryst', 'uchodźc', 'abw$']
zagraniczne_ue: ['dyplomat', 'ambasad', 'nato$']
ustroj_administracja: ['powiat', 'partia$', 'partyjn', 'kadencj', 'posłow', 'stołeczn']
infrastruktura_transport: ['kolej$', 'lotnisk', 'autostrad', 'samochod', 'budyn', 'kataster', 'metro$']
technologie_media: ['teleko', 'internet'

### Krok B7: ocena na zbiorze `tune` (tu strojenie słownika)

Wczytujemy twoje ręczne etykiety z `results/gold_labels.csv` (arkusz z kroku A6 po oznaczeniu). Wiersze bez etykiety i z etykietą spoza listy kategorii są pomijane, z komunikatem.

Miary:
1. `accuracy`: odsetek zgodnych etykiet głównych.
2. `accuracy z etykietą dodatkową`: trafienie liczy się też wtedy, gdy etykieta dodatkowa klasyfikatora zgadza się z twoją główną (wiele tytułów dotyczy dwóch obszarów).
3. `kappa Cohena`: zgodność poprawiona o zgodność przypadkową (1 to zgodność pełna, 0 to poziom przypadku). Liczymy ją ręcznie, żeby uniknąć dodatkowych bibliotek.

In [78]:
GOLD_PATH = RESULTS / "gold_labels.csv"
HAS_GOLD = GOLD_PATH.exists()

if not HAS_GOLD:
    print("Brak pliku", GOLD_PATH)
    print("Oznacz arkusz data/labeling/items_to_label.csv i zapisz go jako results/gold_labels.csv.")
else:
    gold = pd.read_csv(GOLD_PATH, dtype=str)
    gold = gold[["item_id", "split", "label_main"]].dropna(subset=["label_main"])
    n_before = len(gold)
    gold = gold[gold["label_main"].isin(CATEGORY_NAMES)]
    print("Wiersze z poprawną etykietą:", len(gold), "z", n_before)

    # dołączamy predykcje klasyfikatora do ręcznych etykiet
    predictions = items[["item_id", "item", "label", "secondary_label", "confidence"]]
    gold = gold.merge(predictions, on="item_id", how="left")

    # punkty, których item_id nie ma w danych (np. po zmianie reguł normalizacji), pomijamy
    n_missing = int(gold["label"].isna().sum())
    if n_missing > 0:
        print("UWAGA: pomijam", n_missing, "punktów, których item_id nie ma w danych (czy zmieniły się reguły z kroku A2?)")
        gold = gold.dropna(subset=["label"])
    print("Podział:", gold["split"].value_counts().to_dict())

Wiersze z poprawną etykietą: 100 z 100
UWAGA: pomijam 6 punktów, których item_id nie ma w danych (czy zmieniły się reguły z kroku A2?)
Podział: {'test': 57, 'tune': 37}


In [79]:
def cohen_kappa(y_true, y_pred):
    n = len(y_true)
    observed = 0
    for a, b in zip(y_true, y_pred):
        if a == b:
            observed = observed + 1
    observed = observed / n

    expected = 0
    for category in set(y_true) | set(y_pred):
        share_true = y_true.count(category) / n
        share_pred = y_pred.count(category) / n
        expected = expected + share_true * share_pred

    if expected == 1:
        return 1.0
    return (observed - expected) / (1 - expected)


def evaluate(part):
    y_true = list(part["label_main"])
    y_pred = list(part["label"])
    y_second = list(part["secondary_label"])
    n = len(y_true)

    hits = 0
    hits_with_secondary = 0
    for i in range(n):
        if y_pred[i] == y_true[i]:
            hits = hits + 1
            hits_with_secondary = hits_with_secondary + 1
        elif y_second[i] == y_true[i]:
            hits_with_secondary = hits_with_secondary + 1

    print("Liczba punktów:", n)
    print("accuracy:", round(hits / n, 3))
    print("accuracy z etykietą dodatkową:", round(hits_with_secondary / n, 3))
    print("kappa Cohena:", round(cohen_kappa(y_true, y_pred), 3))

In [80]:
if HAS_GOLD:
    tune = gold[gold["split"] == "tune"]
    evaluate(tune)
    print()
    print("Macierz pomyłek (wiersze: ręczna etykieta, kolumny: klasyfikator):")
    display(pd.crosstab(tune["label_main"], tune["label"]))

Liczba punktów: 37
accuracy: 0.919
accuracy z etykietą dodatkową: 1.0
kappa Cohena: 0.908

Macierz pomyłek (wiersze: ręczna etykieta, kolumny: klasyfikator):


label,bezpieczenstwo_obrona,budzet_finanse,edukacja_nauka_kultura,gospodarka_praca,infrastruktura_transport,polityka_spoleczna,rolnictwo_srodowisko,sprawiedliwosc_prawa,ustroj_administracja,zagraniczne_ue,zdrowie
label_main,,,,,,,,,,,
bezpieczenstwo_obrona,3,0,0,0,0,0,0,0,0,0,0
budzet_finanse,0,6,0,0,0,0,0,0,0,0,0
edukacja_nauka_kultura,0,0,6,0,0,0,0,0,0,0,1
gospodarka_praca,0,0,0,2,0,0,0,0,0,0,0
infrastruktura_transport,0,0,0,0,2,0,0,0,0,0,0
polityka_spoleczna,0,0,0,0,0,2,0,0,0,0,0
rolnictwo_srodowisko,0,0,0,2,0,0,2,0,0,0,0
sprawiedliwosc_prawa,0,0,0,0,0,0,0,6,0,0,0
ustroj_administracja,0,0,0,0,0,0,0,0,4,0,0


In [81]:
# błędy na zbiorze tune: dopisz lub usuń słowa kluczowe i uruchom ponownie od kroku B4
if HAS_GOLD:
    errors = tune[tune["label_main"] != tune["label"]]
    print("Liczba błędów:", len(errors))
    for _, row in errors.iterrows():
        print()
        print("ręcznie:", row["label_main"], "| klasyfikator:", row["label"], "| dodatkowa:", row["secondary_label"])
        print(row["item"][:200])

Liczba błędów: 3

ręcznie: rolnictwo_srodowisko | klasyfikator: gospodarka_praca | dodatkowa: rolnictwo_srodowisko
Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o krajowym systemie ewidencji producentów, ewidencji gospodarstw rolnych oraz ewidencji wniosków o przyznanie płatności

ręcznie: rolnictwo_srodowisko | klasyfikator: gospodarka_praca | dodatkowa: rolnictwo_srodowisko
Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o jakości handlowej artykułów rolno-spożywczych oraz niektórych innych ustaw

ręcznie: edukacja_nauka_kultura | klasyfikator: zdrowie | dodatkowa: edukacja_nauka_kultura
Sprawozdanie Komisji o rządowym projekcie ustawy o zmianie ustawy o utworzeniu Uniwersytetu Medycznego w Łodzi


### Krok B8: ocena końcowa na zbiorze `test`

Uruchom tę komórkę dopiero wtedy, gdy skończysz poprawiać słownik. Jeśli po zobaczeniu wyniku wrócisz do słownika, wynik na `test` przestaje być uczciwą oceną, bo słownik jest już dopasowany do tych przykładów. Zapisz wynik i wersję słownika (`RULES_VERSION`) w dokumentacji metody.

In [82]:
if HAS_GOLD:
    test = gold[gold["split"] == "test"]
    evaluate(test)
    print()
    display(pd.crosstab(test["label_main"], test["label"]))

Liczba punktów: 57
accuracy: 0.719
accuracy z etykietą dodatkową: 0.86
kappa Cohena: 0.689



label,bezpieczenstwo_obrona,budzet_finanse,edukacja_nauka_kultura,energia_klimat,gospodarka_praca,infrastruktura_transport,inne,rolnictwo_srodowisko,sprawiedliwosc_prawa,ustroj_administracja,zagraniczne_ue,zdrowie
label_main,,,,,,,,,,,,
bezpieczenstwo_obrona,0,0,0,0,0,0,0,0,0,3,0,0
budzet_finanse,0,6,0,0,0,0,0,0,0,0,0,0
edukacja_nauka_kultura,0,0,6,0,0,0,0,0,0,0,0,0
energia_klimat,0,0,0,2,0,0,1,0,0,0,0,0
gospodarka_praca,0,0,0,0,3,0,1,0,0,0,0,0
infrastruktura_transport,0,0,0,0,0,5,0,0,0,0,0,0
polityka_spoleczna,1,0,0,0,0,0,0,0,0,0,0,0
proceduralne_organizacyjne,0,0,0,0,0,0,0,0,0,1,0,0
rolnictwo_srodowisko,0,0,0,0,1,0,0,2,0,0,0,0


### Krok B9: zapis wyników

Zapisujemy trzy rzeczy: etykiety dla punktów (w formacie, który będzie mógł zastąpić inna metoda, np. model językowy), etykiety i typ głosowania dla wszystkich głosowań oraz kopię słownika z numerem wersji. Głosowania proceduralne (z reguł w kroku A3) dostają etykietę z kolumny `rule_category`.

In [97]:
output = items[["item_id", "label", "secondary_label", "confidence"]].copy()
output["method"] = "rules_spacy"
output["rules_version"] = RULES_VERSION
output.to_csv(INTERIM / "labels_rules_spacy.csv", index=False)

votings_labeled = votings.merge(
    output[["item_id", "label", "secondary_label", "confidence"]], on="item_id", how="left"
)
votings_labeled["label"] = votings_labeled["label"].fillna(votings_labeled["rule_category"])
columns = [
    "glosowanie_id", "sitting", "voting_number", "date", "kind", "vote_type", "stage",
    "item_id", "item", "label", "secondary_label", "confidence",
]
votings_labeled[columns].to_csv(INTERIM / "votings_labeled.csv", index=False)

with open(RESULTS / f"rules_spacy_{RULES_VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(RULES, f, ensure_ascii=False, indent=1)

print("Rozkład etykiet w głosowaniach:")
print(votings_labeled["label"].value_counts(dropna=False))

KeyError: 'item_id'